In [ ]:
import pandas as pd
import numpy as np
import torch

from speos.preprocessing.handler import InputHandler
from speos.utils.config import Config
from speos.preprocessing.datasets import DatasetBootstrapper

In [ ]:
import os
os.chdir("..")

In [ ]:
config = Config()
#config.parse_yaml("config_uc_only_nohetio_film_newstorage.yaml")
config.parse_yaml("config_scz_only_nohetio_film_newstorage.yaml")
#config.parse_yaml("config_cad_really_only_nohetio_film_newstorage.yaml")
prepro = InputHandler(config).get_preprocessor()
prepro.build_graph(adjacency=False)

In [ ]:
from speos.utils.nn_utils import typed_edges_to_sparse_tensor
from torch_sparse import SparseTensor

dataset = DatasetBootstrapper(holdout_size=config.input.holdout_size, name=config.name, config=config).get_dataset()
edge_index, encoder = typed_edges_to_sparse_tensor(dataset.data.x, dataset.data.edge_index_dict)

edge_index_flat = torch.vstack((edge_index.storage.row(), edge_index.storage.col()))
edge_index_flat_reversed = torch.vstack((edge_index.storage.col(), edge_index.storage.row()))
#edge_index_flat = add_remaining_self_loops(edge_index_flat)[0]
edge_index_new = SparseTensor(row = edge_index_flat[0, :], col= edge_index_flat[1, :])

In [ ]:
import json

with open("/mnt/storage/speos/results/scz_film_nohetioouter_results.json", "r") as file:
    results =  [key for key, value in json.load(file)[0].items() if value >= 11]

indices = torch.LongTensor([prepro.hgnc2id[hgnc] for hgnc in results])

with open("/mnt/storage/speos/results/scz_film_nohetioouter_results.json", "r") as file:
    results =  [key for key, value in json.load(file)[0].items() if value >= 1 and value < 11]

indices_weak = torch.LongTensor([prepro.hgnc2id[hgnc] for hgnc in results])

coregenes = dataset.data.y.long() 
coregenes[indices] = 1
coregenes.sum()

In [ ]:
coregenes_hgnc = [prepro.id2hgnc[_id.item()] for _id in coregenes.nonzero()]

In [ ]:
id2entrez = {value: key for key, value in prepro.entrez2id.items()}
coregenes_entrez = [id2entrez[_id.item()] for _id in coregenes.nonzero()]

In [ ]:
id2ensembl = {value: key for key, value in prepro.ensembl2id.items()}
coregenes_ensemble = [id2ensembl[_id.item()] for _id in coregenes.nonzero()]

In [ ]:
df = pd.DataFrame(data={"Symbol" : coregenes_hgnc, "Entrez": coregenes_entrez, "Ensembl": coregenes_ensemble})

df.to_csv("UC_coregenes.csv", index=False)

In [ ]:
import torch

genes = []
edge_attributions = []

for gene in [prepro.id2hgnc[idx.item()] for idx in coregenes.nonzero()]:
    try:
        edge_attributions.append(torch.load("/mnt/storage/speos/explanations/scz_film_nohetio_ig_attr_edge_total_{}.pt".format(gene)).detach().float().cpu().numpy())
        genes.append(gene)
    except (FileNotFoundError, RuntimeError):
        continue

edge_attributions = np.asarray(edge_attributions)
edge_attributions.shape

In [ ]:
import pandas as pd
edge_df = pd.DataFrame({"from": edge_index.storage.row().tolist(),
                        "to": edge_index.storage.col().tolist(),
                        "type":  encoder.inverse_transform(edge_index.storage.value().long().tolist()).tolist(),
                        "importance": edge_attributions.max(axis=0)})

important_edges = {}
hgnc_edge_df = edge_df.copy()
hgnc_edge_df["from"] = [prepro.id2hgnc[_id] for _id in edge_df["from"]]
hgnc_edge_df["to"] = [prepro.id2hgnc[_id] for _id in edge_df["to"]]

hgnc_edge_df.to_csv("disease_network_scz_full.txt", sep="\t", index=False)

for threshold in [0.75, 0.5, 0.25, 0.1, 0.01]:
    for i, gene in enumerate(genes):
        important_indices = (edge_attributions[i, :] > threshold).nonzero()[0]
        important_edges[gene] = important_indices, edge_attributions[i, :][important_indices]

    with open("disease_network_scz_0{}.txt".format(int(threshold*100)), "w") as file:
        for gene, (indices, values) in important_edges.items():
            file.writelines("{}\t{}\t{}\t{}\n".format(prepro.id2hgnc[sender], prepro.id2hgnc[receiver], edgetype, value) for sender, receiver, edgetype, value in zip(edge_df["from"][indices], edge_df["to"][indices], edge_df["type"][indices], values))

In [ ]:
important_edges

In [ ]:
edge_attributions.max(axis=0).shape

In [ ]:
for threshold in [0.75, 0.5, 0.25, 0.1, 0.01]:
    print("disease_network_cad_0{}.txt".format(int(threshold*100)))

In [ ]:
from collections import Counter

count_dfs = []
total_counts = []
num_genes = []

for level in ["75", "50", "25", "10", "1"]:
    disease_edges = pd.read_csv("disease_network_uc_0{}.txt".format(level), sep="\t", index_col=False, header=None, names=["from", "to", "type", "weight"])
    disease_edges = disease_edges.groupby(["from", "to", "type"]).agg("max").reset_index()
    counter = Counter(disease_edges["type"])
    count_df = pd.DataFrame.from_dict(counter, orient="index", columns=[level])
    count_df[level] /= count_df[level].sum()
    count_dfs.append(count_df)
    total_counts.append(len(disease_edges))

    num_genes.append(len(set(disease_edges["to"].tolist()).union(set(disease_edges["from"].tolist()))))

count_dfs = pd.concat(count_dfs, axis=1, join="outer").fillna(0).sort_values(by="75", ascending=True)

count_dfs["0"] = [dataset.data.edge_index_dict[("gene", adj, "gene")].shape[1] for adj in count_dfs.index]
total_counts.append(count_dfs["0"].sum())
count_dfs["0"] /= count_dfs["0"].sum()

num_genes.append(edge_index_flat.flatten().unique().shape[0])



In [ ]:
num_genes

In [ ]:
pretty_names = {
    "BioPlex30293T": "BioPlex 3.0 HEK293T",
    "BioPlex30HCT116": "BioPlex 3.0 HCT116",
    "HuRI": "HuRI",
    'GRNDBadrenalgland': 'GRNDB Adrenal Gland',
    'GRNDBbloodvessel': "GRNDB Blood Vessel",
    'Recon3DDirected': "Recon 3D",
    'GRNDBsalivarygland': "GRNDB Salivary Gland", 
    'GRNDBsmallintestine': "GRNDB Small Intestine", 
    'GRNDButerus': "GRNDB Uterus",
    'GRNDBadiposetissue': 'GRNDB Adipose Tissue', 
    'GRNDBthyroid': 'GRNDB Thyroid', 
    'GRNDBstomach': "GRNDB Stomach", 
    'GRNDBcolon': "GRNDB Colon",
    'GRNDBovary': "GRNDB Ovary", 
    'GRNDBpituitary': "GRNDB Pituitary", 
    'GRNDBesophagus': "GRNDB Esophagus", 
    'GRNDBbrain': "GRNDB Brain",
    'GRNDBliver': "GRNDB Liver", 
    'GRNDBprostate': 'GRNDB Prostate', 
    'GRNDBheart': 'GRNDB Heart', 
    'GRNDBmuscle': 'GRNDB Muscle',
    'GRNDBkidney': "GRNDB Kidney",
    'GRNDBnerve': 'GRNDB Nerve', 
    'GRNDBbreast': "GRNDB Breast", 
    'GRNDBpancreas': "GRNDB Pancreas",
    'GRNDBtestis': "GRNDB Testis", 
    'GRNDBspleen': "GRNDB Spleen", 
    'GRNDBlung': "GRNDB Lung", 
    'GRNDBbloodx': "GRNDB Blood", 
    'GRNDBskin': "GRNDB Skin",
    'GRNDBvagina': "GRNDB Vagina"
}

class ColorCycler:
    def __init__(self, colors):
        self.state = 0
        self.colors = colors

    def next(self):
        color = self.colors[self.state]
        if self.state == len(self.colors) - 1:
            self.state = 0
        else:
            self.state += 1
        return color
    

In [ ]:
from speos.visualization.settings import *
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

#cycler = ColorCycler(["#01016f", "#89006b", "#d00053", "#f85732", "#ffa600"])

x = np.linspace(0.4, 1.0, 14)


cycler1 = ColorCycler(["#000066", "#640069", "#9e0061", "#cc0052", "#eb3a3e", "#fc7225", "#ffa600"][::-1])
cycler2 = ColorCycler(mpl.colormaps["Greens"](x))
cycler3 = ColorCycler(mpl.colormaps["Blues"](x)[::-1])
reconcolor = "#888888"


fig, (ax0, ax) = plt.subplots(nrows=2, figsize=(full_width*cm*0.5, 10*cm), sharex=True, gridspec_kw={'height_ratios': [1, 3]})

running = np.zeros((len(count_dfs.columns),))

ax0.plot(range(len(running)), num_genes, color="gray")
ax0.fill_between(range(len(running)), running, running+np.asarray(num_genes), color="gray", alpha=1)
ax0.set_ylabel("Genes")
ax0.set_ylim(bottom=0)
ax1 = ax0.twinx()
ax1.set_ylabel("% of Total\nNetwork")
ax1.set_ylim((0,1))
ax1.set_yticks((0, 0.2, 0.4, 0.6, 0.8, 1))
ax1.set_yticklabels((0, 20, 40, 60, 80, 100))
ax1.grid(axis="y", zorder=-5, linestyle=":")

grndb_index = 0

for i, (idx, row) in enumerate(count_dfs.iterrows()):
    if "HuRI" in row.name or "BioPlex" in row.name:
        color = cycler1.next()
    elif "Recon" in row.name:
        color = reconcolor
    else:
        color = [cycler2, cycler3][grndb_index % 2].next()
        grndb_index += 1
    #line = ax.plot(range(len(running)), running+row.values, linewidth=1, color=color)
    ax.fill_between(range(len(running)), running, running+row.values, color=color, alpha=1)
    running += row.values
    ax.text(x=5.05, ha="left", y=i/len(count_dfs.index), s=pretty_names[idx], color=color, fontsize=5)

ax.set_xticks((0,1,2,3,4, 5))
ax.set_xticklabels(["{}\n(n={})".format(importance, num_edges) for importance, num_edges in  zip((".75", ".5", ".25", ".1", ".01", "0"), total_counts)])
ax.set_xlim((0,5))
ax.set_ylim((0,1))
ax.set_yticks((0, 0.2, 0.4, 0.6, 0.8, 1))
ax.set_yticklabels((0, 20, 40, 60, 80, 100))
ax.set_xlabel("Attributed Importance (>=)\n(Number of Edges)")
ax.set_ylabel("Percentage of Subnetwork")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.subplots_adjust(wspace=0, hspace=0.05)
#plt.savefig("edge_importance_uc.svg", dpi=450, bbox_inches="tight")

# Print edgetype-frequencies across traits

In [ ]:
from collections import Counter
import pandas as pd

count_dfs = []
total_counts = []
num_genes = []

for trait in ["uc_", "cad_", "scz_"]:
    disease_edges = pd.read_csv("disease_network_{}075.txt".format(trait), sep="\t", index_col=False, header=None, names=["from", "to", "type", "weight"])
    disease_edges = disease_edges.groupby(["from", "to", "type"]).agg("max").reset_index()
    total_counts.append(len(disease_edges))
    counter = Counter(disease_edges["type"])
    count_df = pd.DataFrame.from_dict(counter, orient="index", columns=[trait])
    count_df[trait] /= count_df[trait].sum()
    count_dfs.append(count_df)

In [ ]:
count_df = pd.concat(count_dfs, axis=1)

In [ ]:
count_df.columns = ["UC", "CAD", "SCZ"]
count_df = count_df.fillna(0)


In [ ]:
total = edge_index_flat.shape[1]
type2frequency = {key[1]: values.shape[1] / total for key, values in dataset.data.edge_index_dict.items()}

In [ ]:
grndb_sorted = sorted([index for index in count_df.index.tolist() if "GRNDB" in index])

ppis = ["BioPlex30293T", "BioPlex30HCT116", "HuRI"]

metabolic = ["Recon3DDirected"]

In [ ]:
count_df = count_df.loc[ppis + metabolic + grndb_sorted, :]

In [ ]:
import json
from extensions.preprocessing import preprocess_labels
from torch_geometric.utils import k_hop_subgraph

trait = "uc"

def get_coregenes(trait: str):
    trait2name = {"uc": "uc",
                "cad": "cad_really",
                "scz": "scz",
                "ad": "alz",
                "ra": "ra"}

    config = Config()
    #config.parse_yaml("config_cad_really_only_nohetio_film_newstorage.yaml")
    config.parse_yaml("config_{}_only_nohetio_film_newstorage.yaml".format(trait2name[trait]))
    prepro = InputHandler(config).get_preprocessor()
    prepro.build_graph(adjacency=False)

    background = prepro.id2hgnc.keys()

    mendelians = prepro.pos_idx

    with open("/mnt/storage/speos/results/{}_film_nohetioouter_results.json".format(trait2name[trait]), "r") as file:
        candidate2cs = json.load(file)[0]

    coregenes = [prepro.hgnc2id[key] for key, value in candidate2cs.items() if value == 11]

    allcore = set()
    allcore.update(set(coregenes))
    allcore.update(set(mendelians))

    dataset = DatasetBootstrapper(holdout_size=config.input.holdout_size, name=config.name, config=config).get_dataset()

    total_incident_edges = 0
    type2localfrequency = {}
    for key, edges in dataset.data.edge_index_dict.items():
        _, incident_edges, _, _ = k_hop_subgraph(list(allcore), 1, edges, flow="source_to_target", directed=True)
        total_incident_edges += incident_edges.shape[1]
        type2localfrequency[key[1]] = incident_edges.shape[1]
    
    for key in type2localfrequency.keys():
        type2localfrequency[key] /= total_incident_edges
     
    return allcore, type2localfrequency


trait2coregenes = {trait: get_coregenes(trait) for trait in ["uc", "cad", "scz"]}

In [ ]:
from speos.visualization.settings import *
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binomtest
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import binom

def pval_to_string(pval):
    if pval < 0.05:
        return "*"
    else:
        return ""

fig, ax = plt.subplots(figsize=(full_width*cm, 4*cm))
handles = []
handles.append(ax.scatter(np.arange(len(count_df["UC"]))-0.25, count_df["UC"], marker="^", edgecolor="black",linewidth=0, facecolor="teal", s=8))
handles.append(ax.scatter(np.arange(len(count_df["CAD"])), count_df["CAD"], marker="p", edgecolor="black",linewidth=0, facecolor="crimson", s=8))
handles.append(ax.scatter(np.arange(len(count_df["SCZ"]))+0.25, count_df["SCZ"], marker="o", edgecolor="black",linewidth=0, facecolor="#fcae1e", s=8))
ax.set_xticks(range(len(ppis + metabolic + grndb_sorted)))
ax.set_xticklabels([pretty_names[name] for name in ppis + metabolic + grndb_sorted], rotation=90, fontsize=8)

cadpvals = []
cadstats = []
sczpvals = []
sczstats = []
midtops = []

typelist = []
traitslist = []
toplist= []
midlist = []
bottomlist = []
for i, type in enumerate(ppis + metabolic + grndb_sorted):
    for j, (trait, (genes, type2localfrequency)) in enumerate(trait2coregenes.items()):
        line = ax.hlines((type2localfrequency[type]), xmin=-0.375 + (j *0.25) + i, xmax = -0.175 + (j * 0.25) + i, color="black", linewidth=0.5)
        bottom = binom.ppf(0.025, n=total_counts[j], p=type2localfrequency[type]) / total_counts[j]
        mid = type2localfrequency[type]
        top = binom.ppf(0.975, n=total_counts[j], p=type2localfrequency[type]) / total_counts[j]
        if j == 1:
            midtops.append(top)
        bar = ax.bar(x=(-0.25 + j*0.25) + i,width=0.25, height= top-mid, bottom=mid, color="lightgrey", edgecolor="darkgrey", linewidth=0.1, zorder=-1)
        #gradientbars(bar)
        bar = ax.bar(x=(-0.25 + j*0.25) + i,width=0.25, height= mid-bottom, bottom=bottom, color="lightgrey", edgecolor="darkgrey", linewidth=0.1, zorder=-1)

        if trait == "uc":
            uc = count_df.loc[type, count_df.columns[j]]
            ucj = j
        elif trait == "cad":
            cad = count_df.loc[type, count_df.columns[j]]
            cadj = j
        elif trait == "scz":
            scz = count_df.loc[type, count_df.columns[j]]
            sczj = j

        typelist.append(type)
        traitslist.append(trait)
        toplist.append(top)
        midlist.append(mid)
        bottomlist.append(bottom)
    
    cadstats.append(binomtest(int(uc * total_counts[ucj]), n=total_counts[ucj], p=cad, alternative='two-sided').statistic)
    cadpvals.append(binomtest(int(uc * total_counts[ucj]), n=total_counts[ucj], p=cad, alternative='two-sided').pvalue)
    sczstats.append(binomtest(int(uc * total_counts[ucj]), n=total_counts[ucj], p=scz, alternative='two-sided').statistic)
    sczpvals.append(binomtest(int(uc * total_counts[ucj]), n=total_counts[ucj], p=scz, alternative='two-sided').pvalue)

cadfdr = fdrcorrection(cadpvals)[1]
sczfdr = fdrcorrection(sczpvals)[1]
for i, (x, p) in enumerate(zip(midtops, cadfdr)):
    if p < 0.05:
        ax.text(s=pval_to_string(p), x=i, y=x+0.005, ha="center", va="bottom", fontsize=5)

for i, (x, p) in enumerate(zip(midtops, sczfdr)):
    if p < 0.05:
        ax.text(s=pval_to_string(p), x=i +0.25, y=x+0.005, ha="center", va="bottom", fontsize=5)



handles.append(line)
handles.append(bar)
#for i in range(len(count_df)):
#    ax.fill_between((-0.5 + i, 0.5 + i), -1, 1, facecolor=["lightgrey", "grey"][i%2], alpha=0.3, zorder=-1)#

#for i in range(len(ppis)):
#    ax.fill_between((-0.5 + i, 0.5 + i), -1, 1, facecolor=["lightgreen", "green"][i%2], alpha=0.3, zorder=-1)#

#ax.fill_between((0.5 + i, 1.5 + i), -1, 1,  facecolor="grey", alpha=0.3, zorder=-1)

#for i in range(len(grndb_sorted)):
#    ax.fill_between((3.5 + i, 4.5 + i), -1, 1, facecolor=["lightblue", "blue"][i%2], alpha=0.3, zorder=-1)
ax.legend(handles, ["UC", "CAD", "SCZ", "Exp. Frequency", "95% CI (Binom.)"], fontsize=6, markerscale=2)
ax.vlines(x=np.arange(len(count_df) + 1) - 0.5, ymin=0, ymax=0.35, linestyles=":", color="#dddddd", linewidth=0.4)

ax.hlines(y=0.35, xmin=-0.25, xmax=2.4, linestyles="--", color="grey", linewidth=1)
ax.hlines(y=0.35, xmin=2.6, xmax=3.4, linestyles="--", color="grey", linewidth=1)
ax.hlines(y=0.35, xmin=3.6, xmax=len(count_df)+0.5, linestyles="--", color="grey", linewidth=1)
ax.vlines(x=2.4, ymin=0.3, ymax=0.35, linestyles="--", color="grey", linewidth=1)
ax.vlines(x=3.4, ymin=0.3, ymax=0.35, linestyles="--", color="grey", linewidth=1)
#ax.vlines(x=len(count_df)-0.5, ymin=0.3, ymax=0.35, linestyles="--", color="black", linewidth=1)

ax.text(va="top", ha="right", x=2.1, y=0.34, s="PPI", color="grey", fontsize=8)
ax.text(va="top", ha="center", x=3, y=0.34, s="Metabolic", color="grey", rotation=90, fontsize=8)
ax.text(va="top", ha="left", x=3.8, y=0.34, s="Regulatory", color="grey", fontsize=8)

ax.set_ylim(0, 0.35)
ax.set_xlim(-0.5, len(count_df)-0.5)
ax.set_ylabel("Rel. Freq. of Important Edges")
ax.set_xlabel("Subnetwork")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.savefig("edge_comparison_local_ucvsall.svg", bbox_inches="tight", dpi=450)


In [ ]:
df = pd.DataFrame({
    "Network": typelist,
    "Trait": traitslist,
    "Expected Frequency": midlist,
    "CI95 upper": toplist,
    "CI95 lower": bottomlist
})

In [ ]:
observed = []
for network, trait in zip(df.Network, df.Trait):
    observed.append(count_df.loc[network, trait.upper()])

In [ ]:
df["Observed Frequency"] = observed

In [ ]:
allpvals = []
allFDR = []
allstats = []
for i in range(len(cadpvals)):
    allstats.append(None)
    allstats.append(cadstats[i])
    allstats.append(sczstats[i])
    allpvals.append(None)
    allpvals.append(cadpvals[i])
    allpvals.append(sczpvals[i])
    allFDR.append(None)
    allFDR.append(cadfdr[i])
    allFDR.append(sczfdr[i])

In [ ]:
df["Binom. Test statistic"] = allstats
df["Binom. Test p-val"] = allpvals
df["Binom. Test FDR"] = allFDR

In [ ]:
allFDR

In [ ]:
df.to_csv("network_frequencies_tests.tsv", sep="\t", index=False, float_format='%.15f')

# Getting important metabolic connections in CAD

In [ ]:

edges_cad = pd.read_csv("disease_network_cad_075.txt", header=None, sep="\t", names=["sender", "receiver", "type", "weight"])

In [ ]:
metabolic_edges_cad = edges_cad[edges_cad.type == "Recon3DDirected"]

In [ ]:
metabolic_edges_cad.reset_index(inplace=True, drop=True)

In [ ]:
metabolic_edges_cad

In [ ]:
recon = pd.read_csv("/mnt/storage/speos/data/recon/reconparser/data/recon_directed_absolute.tsv", header=0, sep="\t")

In [ ]:
recon = recon.replace({"EntrezA": prepro.entrez2id, "EntrezB": prepro.entrez2id})
recon = recon.replace({"EntrezA": prepro.id2hgnc, "EntrezB": prepro.id2hgnc})

In [ ]:
recon["Metabolite"][(recon["EntrezA"] == metabolic_edges_cad.iloc[0,:]["sender"])]

In [ ]:
translated_edges = []
for i, row in metabolic_edges_cad.iterrows():
    metabolites = recon["Metabolite"][(recon["EntrezA"] == metabolic_edges_cad.iloc[i,:]["sender"]) & (recon["EntrezB"] == metabolic_edges_cad.iloc[i,:]["receiver"])]
    senders = [row["sender"]] * len(metabolites)
    receivers = [row["receiver"]] * len(metabolites)
    weight = [row["weight"]] * len(metabolites)
    df = pd.DataFrame({"Sender": senders,
                       "Receiver": receivers,
                       "Importance": weight,
                       "Metabolite": metabolites})
    translated_edges.append(df)

In [ ]:
dfs = pd.concat(translated_edges)

In [ ]:
dfs.sort_values("Importance", ascending=False).reset_index(drop=True)

In [ ]:
dfs.to_csv("important_metabolic_connections_cad.tsv", sep="\t", index=False)